# Impact of External Ressource on Detection Performance

In [1]:
import sys
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import combinations

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR, EXP_COLS, CLASS_COLS
from utils.analysis_helpers import wrap_model_comparison, wrap_model_comparison_union, print_model_comparison_results, combine_model_comparison_outputs, extract_metric_values, compute_balanced_metrics

In [2]:
PROVIDER="gpt"

In [3]:
df_b_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_1.feather"))
df_d_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_1.feather"))
df_b_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_0.feather"))
df_d_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_0.feather"))

In [4]:
df_d_1.columns

Index(['comment_cleaned', 'id', 'discourse', 'comment_level',
       'comment_codes_all', 'source_outlet', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'explanation_ihra_explanation_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'comment_codes_all_list',
       'comment_codes_all_chapters', 'comment_codes_all_sections'],
      dtype='object')

In [5]:
df_b_1.columns

Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_tax', 'explanation_tax', 'classification_tax_ex',
       'explanation_tax_ex', 'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'explanation_ihra_explanation_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters'],
      dtype='object')

In [6]:
cols_d = ['comment_cleaned', 'discourse', 'comment_level', 
          'comment_codes_all', 'source_outlet'] + EXP_COLS + CLASS_COLS

for c in cols_d: 
    if c not in df_d_0.columns:
        print(c)

In [7]:
cols_b = ['comment_cleaned', 'keyword', 'ihra_section_1', 'ihra_section_2'] + EXP_COLS + CLASS_COLS

for c in cols_b: 
    if c not in df_b_0.columns:
        print(c)

In [8]:
N_neg_total_d = 39424
N_neg_total_b = 9358

### Merge positive and negative samples

In [9]:
df_d_0["label"] = 0
df_d_1["label"] = 1
df_b_0["label"] = 0
df_b_1["label"] = 1

cols_b.insert(0,"label")
cols_d.insert(0,"label")

In [10]:
df_b = pd.concat([df_b_0[cols_b], df_b_1[cols_b]], ignore_index=True)
df_d = pd.concat([df_d_0[cols_d], df_d_1[cols_d]], ignore_index=True)

In [11]:
df_b.label.value_counts()

label
1    1858
0    1000
Name: count, dtype: int64

In [12]:
df_d.label.value_counts()

label
1    2977
0     992
Name: count, dtype: int64

### Combine datasets

In [14]:
w_neg_d = N_neg_total_d / len(df_d_0)
w_neg_b = N_neg_total_b / len(df_b_0)

df_d["dataset_id"] = "d"
df_b["dataset_id"] = "b"

df_union = pd.concat([df_d, df_b], ignore_index=True)

N_neg_total_by_dataset = {
    "d": N_neg_total_d,
    "b": N_neg_total_b,
}

### Distribution of responses in the positive samples

In [15]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.000538,NaN,NaN,NaN
No,0.489774,0.479548,0.329386,0.315393
Unsure,0.000538,0.032831,0.035522,0.039828
Yes,0.509150,0.487621,0.635091,0.644779


In [16]:
# Decoding  
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.004367,NaN,NaN,NaN
No,0.733289,0.719516,0.464898,0.404434
Unsure,0.005375,0.067182,0.057776,0.045348
Yes,0.256970,0.213302,0.477326,0.550218


### Distribution of responses in the negative samples

In [17]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.005,NaN,NaN,NaN
No,0.918,0.938,0.924,0.917
Unsure,0.002,0.028,0.022,0.024
Yes,0.075,0.034,0.054,0.059


In [18]:
# Decoding
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.040323,NaN,NaN,NaN
No,0.940524,0.807460,0.842742,0.812500
Unsure,0.007056,0.189516,0.147177,0.169355
Yes,0.012097,0.003024,0.010081,0.018145


### Compute metrics and CIs on weighted union of both datasets

- B and D have the same weight (0.5)
- CI for recall reflects sampling from population; Ci for precision and F1 reflects sampling from population and 1000er sampling from dataset, thus larger CIs

In [19]:
model_comparison_metrics = compute_balanced_metrics(df_B=df_b, df_D=df_d, N_neg_total_B=N_neg_total_b, N_neg_total_D=N_neg_total_d)
model_comparison_metrics

,model,metric,value,ci_low,ci_high
0,no_kb,precision,0.595036,0.529682,0.678482
1,no_kb,recall,0.383060,0.369296,0.397299
2,no_kb,f1,0.451161,0.430223,0.472781
3,ihra,precision,0.791008,0.714970,0.874517
4,ihra,recall,0.350462,0.336921,0.364552
5,ihra,f1,0.464134,0.446100,0.482830
6,tax,precision,0.740803,0.684713,0.804813
7,tax,recall,0.556209,0.541774,0.570197
8,tax,f1,0.629344,0.606910,0.652138
9,tax_ex,precision,0.690276,0.638388,0.752455


In [20]:
model_comparison_metrics.to_parquet(f"{PROVIDER}_model_comparison_metrics.parquet", index=False)

### Export errors for qualitative analysis

In [18]:
df_b[(df_b["label"]==0) & (df_b["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_decoding.csv"), index=False)
df_b[(df_b["label"]==0) & (df_b["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_decoding.csv"), index=False)

### Statistical model comparison vs baseline (no_kb)

## Bloomington

Mapping "Yes" to 1 and Unsure to 0

1. class proportion as in the original dataset
2. class balance

In [19]:
res_b_weighted = wrap_model_comparison(df=df_b, N_neg_total=N_neg_total_b)
print_model_comparison_results(res_b_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.7401
metric(no_kb): 0.5741
Diff (model - baseline): 0.1660
95% CI: [0.0819, 0.2497]
p-value (raw): 0.0001
p-value (adjusted): 0.0003
Effect size h: 0.3522 (95% CI: [0.1723, 0.5396])

Model: tax
metric(tax): 0.7002
metric(no_kb): 0.5741
Diff (model - baseline): 0.1261
95% CI: [0.0507, 0.2005]
p-value (raw): 0.0009
p-value (adjusted): 0.0018
Effect size h: 0.2632 (95% CI: [0.1059, 0.4250])

Model: tax_ex
metric(tax_ex): 0.6845
metric(no_kb): 0.5741
Diff (model - baseline): 0.1104
95% CI: [0.0340, 0.1830]
p-value (raw): 0.003
p-value (adjusted): 0.003
Effect size h: 0.2293 (95% CI: [0.0709, 0.3835])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.4876
metric(no_kb): 0.5091
Diff (model - baseline): -0.0215
95% CI: [nan, nan]
p-value (raw): 0.1998
p-value (adjusted): 0.1998
Effect size h: -0.0431 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.6351
metric(no_kb): 0.5091
Diff (model - baseline): 0.1259
95% CI: [nan, nan]


### Decoding

Mapping "Yes" to 1 and "Unsure" to 0

1. class proportion as in the original dataset
2. class balance

In [21]:
res_d_weighted = wrap_model_comparison(df=df_d, N_neg_total=N_neg_total_d)
print_model_comparison_results(res_d_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.8419
metric(no_kb): 0.6160
Diff (model - baseline): 0.2259
95% CI: [0.0179, 0.4204]
p-value (raw): 0.0328
p-value (adjusted): 0.0984
Effect size h: 0.5189 (95% CI: [0.0425, 1.3341])

Model: tax
metric(tax): 0.7814
metric(no_kb): 0.6160
Diff (model - baseline): 0.1655
95% CI: [-0.0199, 0.3289]
p-value (raw): 0.0518
p-value (adjusted): 0.1036
Effect size h: 0.3638 (95% CI: [-0.0446, 0.7498])

Model: tax_ex
metric(tax_ex): 0.6960
metric(no_kb): 0.6160
Diff (model - baseline): 0.0800
95% CI: [-0.0939, 0.2404]
p-value (raw): 0.3406
p-value (adjusted): 0.3406
Effect size h: 0.1688 (95% CI: [-0.2085, 0.5149])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.2133
metric(no_kb): 0.2570
Diff (model - baseline): -0.0437
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.1031 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.4773
metric(no_kb): 0.2570
Diff (model - baseline): 0.2204
95% CI: [nan, nan]
p-va

### Combined datasets

In [23]:
# Union of both datasets is treated according to the class distribution in the original datasets and according to the overall dataset size, thus decoding matters more
res_union_weighted = wrap_model_comparison_union(df_union, N_neg_total_by_dataset)
print_model_comparison_results(res_union_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.7789
metric(no_kb): 0.5921
Diff (model - baseline): 0.1868
95% CI: [0.0937, 0.2778]
p-value (raw): 0.0002
p-value (adjusted): 0.0006
Effect size h: 0.4065 (95% CI: [0.2016, 0.6175])

Model: tax
metric(tax): 0.7423
metric(no_kb): 0.5921
Diff (model - baseline): 0.1503
95% CI: [0.0619, 0.2376]
p-value (raw): 0.0002
p-value (adjusted): 0.0006
Effect size h: 0.3208 (95% CI: [0.1341, 0.5131])

Model: tax_ex
metric(tax_ex): 0.6911
metric(no_kb): 0.5921
Diff (model - baseline): 0.0990
95% CI: [0.0114, 0.1860]
p-value (raw): 0.0224
p-value (adjusted): 0.0224
Effect size h: 0.2070 (95% CI: [0.0240, 0.3906])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.3187
metric(no_kb): 0.3539
Diff (model - baseline): -0.0352
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.0744 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.5380
metric(no_kb): 0.3539
Diff (model - baseline): 0.1841
95% CI: [nan, nan]
p-value 

In [24]:
# Union of both datasets is treated according to the class distribution in the original datasets but both datasets weightes equally

res_union_dataset_balanced = combine_model_comparison_outputs(
    {
        "B": res_b_weighted,
        "D": res_d_weighted,
    },
    dataset_weights={
        "B": 0.5,
        "D": 0.5,
    },
)


In [25]:
print_model_comparison_results(res_union_dataset_balanced)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.7910
metric(no_kb): 0.5950
Diff (model - baseline): 0.1960
95% CI: [nan, nan]
p-value (raw): 0.0328
p-value (adjusted): 0.0984
Effect size h: 0.4355 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.7408
metric(no_kb): 0.5950
Diff (model - baseline): 0.1458
95% CI: [nan, nan]
p-value (raw): 0.0518
p-value (adjusted): 0.1036
Effect size h: 0.3135 (95% CI: [nan, nan])

Model: tax_ex
metric(tax_ex): 0.6903
metric(no_kb): 0.5950
Diff (model - baseline): 0.0952
95% CI: [nan, nan]
p-value (raw): 0.3406
p-value (adjusted): 0.3406
Effect size h: 0.1990 (95% CI: [nan, nan])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.3505
metric(no_kb): 0.3831
Diff (model - baseline): -0.0326
95% CI: [nan, nan]
p-value (raw): 0.1998
p-value (adjusted): 0.1998
Effect size h: -0.0731 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.5562
metric(no_kb): 0.3831
Diff (model - baseline): 0.1731
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted

In [26]:
res_union_dataset_balanced

[('precision',
  {'ihra': {'metric_A': 0.7910075037189976,
    'metric_B': 0.5950357022406125,
    'diff': 0.19597180147838505,
    'effect_size_h': np.float64(0.4355311526089536),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.0328),
    'p_value_adj': np.float64(0.09840000000000002)},
   'tax': {'metric_A': 0.7408033446916291,
    'metric_B': 0.5950357022406125,
    'diff': 0.14576764245101653,
    'effect_size_h': np.float64(0.31346277609510975),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.0518),
    'p_value_adj': np.float64(0.1036)},
   'tax_ex': {'metric_A': 0.6902756706707873,
    'metric_B': 0.5950357022406125,
    'diff': 0.09523996843017474,
    'effect_size_h': np.float64(0.199012749755202),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    '

In [27]:
df_B = extract_metric_values(res_b_weighted, "B")
df_D = extract_metric_values(res_d_weighted, "D")
df_union = extract_metric_values(res_union_weighted, "union")
df_union_bal = extract_metric_values(res_union_dataset_balanced, "union_dataset_balanced")

df_metrics = pd.concat(
    [df_B, df_D, df_union, df_union_bal],
    ignore_index=True,
)

In [28]:
df_metrics

,setting,metric,model,value
0,B,precision,no_kb,0.574081
1,B,precision,ihra,0.740092
2,B,precision,tax,0.700159
3,B,precision,tax_ex,0.684524
4,B,recall,no_kb,0.509150
5,B,recall,ihra,0.487621
6,B,recall,tax,0.635091
7,B,recall,tax_ex,0.644779
8,B,f1,no_kb,0.539669
9,B,f1,ihra,0.587897


##  Prediction overlap between models

In [29]:
def intersection_over_union(y_preds, label=1):
    y_preds = [np.asarray(y) for y in y_preds]
    intersection = np.logical_and.reduce([y == label for y in y_preds])
    union = np.logical_or.reduce([y == label for y in y_preds])
    if np.sum(union) == 0:
        return np.nan  # avoid division by zero
    return np.round(np.sum(intersection) / np.sum(union), 2)

### Bloomington

In [30]:
# all 4
intersection_over_union([df_b[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.26)

In [31]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.31
IoU for classification_no_kb_cleaned vs classification_tax: 0.37
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.38
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.73
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.71
IoU for classification_tax vs classification_tax_ex: 0.84


#### Positive samples only

In [32]:
intersection_over_union([
    df_b[df_b.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.28)

In [33]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[df_b.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[df_b.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.33
IoU for classification_no_kb_cleaned vs classification_tax: 0.4
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.4
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.74
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.72
IoU for classification_tax vs classification_tax_ex: 0.85


### Decoding

In [34]:
# all 4
intersection_over_union([df_d[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.07)

In [35]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.12
IoU for classification_no_kb_cleaned vs classification_tax: 0.19
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.21
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.43
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.38
IoU for classification_tax vs classification_tax_ex: 0.72


#### Positive samples only

In [36]:
# all 5
intersection_over_union([df_d[df_d.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.07)

In [37]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[df_d.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[df_d.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.13
IoU for classification_no_kb_cleaned vs classification_tax: 0.19
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.22
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.43
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.38
IoU for classification_tax vs classification_tax_ex: 0.73
